# Hyperparameter Transfer in GNNs — CiteSeer Demo

Companion notebook for **Hyperparameter Transfer in Graph Neural Networks** (LoG Conference, proceedings track).

**Claim in one sentence:** under a transfer parameterization, a learning rate tuned on a *small* residual GCN stays near-optimal when you scale width; under standard parameterization, it does not.

We reproduce that contrast on **CiteSeer** (Planetoid node classification) with Adam and a residual GCN — the architecture used in the paper’s base transfer model (encoder → $L$ MPNN+MLP residual blocks → decoder).

> Runtime: Colab **GPU**, ~5–10 minutes for the default settings below.


In [ ]:
# Cell 1 — install (Colab) and import the public package
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get(
    "GNN_TRANSFER_REPO",
    "https://github.com/YOUR_USERNAME/hp-transfer-gnns.git",  # update after you push
)

if IN_COLAB:
    # PyG wheels track the installed torch build
    import torch
    torver = torch.__version__.split("+")[0]
    cuda = "cu121" if torch.cuda.is_available() else "cpu"
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "matplotlib", "tqdm",
        "torch-geometric",
    ])
    root = pathlib.Path("/content/hp-transfer-gnns")
    if not root.exists():
        # Prefer a public clone; fall back to uploading the repo zip if private.
        try:
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(root)])
        except subprocess.CalledProcessError:
            raise SystemExit(
                "Clone failed. In Colab: Runtime → upload `hp-transfer-gnns` "
                "or set GNN_TRANSFER_REPO to your public URL."
            )
    sys.path.insert(0, str(root / "src"))
else:
    # Local: editable install or src/ on path
    here = pathlib.Path.cwd()
    for candidate in [here / "src", here.parent / "src", pathlib.Path("/home/jdezoort/hp-transfer-gnns/src")]:
        if (candidate / "gnn_transfer").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

import torch
from torch_geometric.datasets import Planetoid
from gnn_transfer import SimpleGNN, StandardGCN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device={device} | cuda={torch.cuda.is_available()}")
assert torch.cuda.is_available() or not IN_COLAB, "Enable a GPU runtime (Runtime → Change runtime type)."


## Transfer recipe (Adam, residual GCN)

The paper’s base model is a residual GCN: degree-normalized message passing, then a $4\times$ MLP, each residual branch scaled so the *total* residual contribution stays $O(1)$ in depth.

For **Adam**, the transferable base learning rate $\eta_0$ maps to a global LR

$$
\eta = \frac{\eta_0}{\sqrt{D}},\qquad \sigma_0 = \sigma_{L+1} = \frac{1}{\sqrt{D}},\qquad \text{residual branch} \propto \frac{1}{L}.
$$

`SimpleGNN` implements that recipe (`get_lr`, encoder/decoder init, $1/L$ residuals).  
`StandardGCN` is the same graph of ops under ordinary fan-in init and $\eta=\eta_0$ with $L^{-1/2}$ residuals — i.e. **no** transfer machinery.

**Experiment design:** sweep $\eta_0$ at proxy width $D=64$, pick the best $\eta_0^\star$, then train $D=256$ at that same $\eta_0^\star$ for both parameterizations.


In [ ]:
# Cell 3 — CiteSeer + full-batch train / eval helpers
DATA_ROOT = pathlib.Path("./data/Planetoid")
dataset = Planetoid(root=str(DATA_ROOT), name="CiteSeer")
data = dataset[0].to(device)
in_dim, n_classes = dataset.num_node_features, dataset.num_classes
print(f"CiteSeer | N={data.num_nodes} E={data.num_edges} F={in_dim} C={n_classes}")

DEPTH = 4          # L — joint width/depth scaling in the paper; keep fixed here for clarity
EPOCHS = 100       # enough to see LR sensitivity without a long Colab wait
ETA0_GRID = torch.logspace(-2.5, 0.5, 9).tolist()  # ~3e-3 … ~3
PROXY_D, WIDE_D = 64, 256
SEED = 0


def accuracy(logits, y, mask):
    pred = logits[mask].argmax(dim=-1)
    return (pred == y[mask]).float().mean().item()


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data)
    return {
        "train": accuracy(logits, data.y, data.train_mask),
        "val": accuracy(logits, data.y, data.val_mask),
        "test": accuracy(logits, data.y, data.test_mask),
    }


def build_model(kind: str, width: int):
    cls = SimpleGNN if kind == "transfer" else StandardGCN
    return cls(
        in_dim=in_dim,
        out_dim=n_classes,
        width=width,
        depth=DEPTH,
        heads=1,
        res_agg=1.0,          # fully use the message-passing residual mix
        optimizer="adam",
        encoder="linear",
        decoder="linear",
        adjacency="normalized",
        pool=False,           # node classification
        layer_norm=False,
    ).to(device)


def train_once(kind: str, width: int, eta0: float, epochs: int = EPOCHS, seed: int = SEED):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = build_model(kind, width)
    # Transfer: η = η0 / √D inside make_param_groups. Standard: η = η0.
    groups = model.make_param_groups(eta_0=eta0, weight_decay=0.0)
    opt = torch.optim.Adam(groups)
    best_val, best_state, history = -1.0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(data)
        loss = torch.nn.functional.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()
        metrics = evaluate(model, data)
        history.append({"epoch": epoch, "loss": float(loss.detach()), **metrics})
        if metrics["val"] > best_val:
            best_val = metrics["val"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    final = evaluate(model, data)
    lr_used = groups[0]["lr"]
    return {"kind": kind, "width": width, "eta0": eta0, "lr": lr_used, **final, "history": history}


In [ ]:
# Cell 4 — proxy sweep at D=64: where does η0* sit for each parameterization?
from tqdm.auto import tqdm

proxy_results = []
for kind in ("transfer", "standard"):
    for eta0 in tqdm(ETA0_GRID, desc=f"proxy D={PROXY_D} [{kind}]"):
        proxy_results.append(train_once(kind, PROXY_D, eta0))

def best_by_val(rows, kind):
    subset = [r for r in rows if r["kind"] == kind]
    return max(subset, key=lambda r: r["val"])

best_transfer = best_by_val(proxy_results, "transfer")
best_standard = best_by_val(proxy_results, "standard")
print(
    f"transfer  η0*={best_transfer['eta0']:.4g}  "
    f"(lr={best_transfer['lr']:.4g})  val={best_transfer['val']:.3f}  test={best_transfer['test']:.3f}"
)
print(
    f"standard  η0*={best_standard['eta0']:.4g}  "
    f"(lr={best_standard['lr']:.4g})  val={best_standard['val']:.3f}  test={best_standard['test']:.3f}"
)


In [ ]:
# Cell 5 — scale to D=256 at the *proxy* optima (no retuning)
# Plus a diagnostic: standard model also tried at the transfer η0* (usually too aggressive / too shy after width change).
wide_runs = []
configs = [
    ("transfer", best_transfer["eta0"], "transfer @ its η0*"),
    ("standard", best_standard["eta0"], "standard @ its η0*"),
    ("standard", best_transfer["eta0"], "standard @ transfer η0* (no retune)"),
]
for kind, eta0, tag in configs:
    print(f"→ wide D={WIDE_D}: {tag}")
    row = train_once(kind, WIDE_D, eta0)
    row["tag"] = tag
    wide_runs.append(row)
    print(f"   val={row['val']:.3f}  test={row['test']:.3f}  lr={row['lr']:.4g}")


In [ ]:
# Cell 6 — figures: LR transfer curves + scale-up summary
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), constrained_layout=True)

# (left) validation vs η0 at proxy width
ax = axes[0]
for kind, color, label in [
    ("transfer", "#1b4f72", "Transfer (SimpleGNN)"),
    ("standard", "#922b21", "Standard (StandardGCN)"),
]:
    rows = sorted([r for r in proxy_results if r["kind"] == kind], key=lambda r: r["eta0"])
    xs = [r["eta0"] for r in rows]
    ys = [r["val"] for r in rows]
    ax.plot(xs, ys, marker="o", color=color, label=label, lw=2)
ax.axvline(best_transfer["eta0"], color="#1b4f72", ls="--", alpha=0.5)
ax.axvline(best_standard["eta0"], color="#922b21", ls="--", alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel(r"base learning rate $\eta_0$")
ax.set_ylabel("validation accuracy")
ax.set_title(f"Proxy sweep (CiteSeer, $D={PROXY_D}$, $L={DEPTH}$)")
ax.legend(frameon=False)
ax.grid(True, which="both", ls=":", alpha=0.5)

# (right) test accuracy: proxy best vs wide at transferred η0*
ax = axes[1]
labels, transfer_bar, standard_bar = [], [], []
# proxy peaks
labels.append(f"proxy $D={PROXY_D}$\n@ own $\\eta_0^\\star$")
transfer_bar.append(best_transfer["test"])
standard_bar.append(best_standard["test"])
# wide at own η0*
wide_t = next(r for r in wide_runs if r["kind"] == "transfer")
wide_s = next(r for r in wide_runs if r["tag"].startswith("standard @ its"))
labels.append(f"wide $D={WIDE_D}$\n@ own $\\eta_0^\\star$")
transfer_bar.append(wide_t["test"])
standard_bar.append(wide_s["test"])

x = np.arange(len(labels))
w = 0.36
ax.bar(x - w / 2, transfer_bar, width=w, color="#1b4f72", label="Transfer")
ax.bar(x + w / 2, standard_bar, width=w, color="#922b21", label="Standard")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("test accuracy (best val ckpt)")
ax.set_ylim(0.0, 1.0)
ax.set_title("Scale-up at proxy-tuned $\\eta_0$")
ax.legend(frameon=False)
ax.grid(True, axis="y", ls=":", alpha=0.5)

fig.suptitle("Hyperparameter transfer vs standard parameterization on CiteSeer", fontsize=12)
plt.show()

print("\nTakeaway")
print("--------")
print(
    "Transfer: η0* chosen at D=64 is reused at D=256 via η=η0/√D; "
    f"test {best_transfer['test']:.3f} → {wide_t['test']:.3f}."
)
print(
    "Standard: the optimizer sees η=η0 with no √D rule, so width changes the "
    f"effective step size; proxy η0* need not remain optimal "
    f"(test {best_standard['test']:.3f} → {wide_s['test']:.3f})."
)
print(
    "That is the practical point of the paper: tune once on a cheap proxy, "
    "scale the GNN without a second LR search."
)
